# Shown Space Scoring Path Visuals

This notebook uses Shown Space's public game API to recreate field-path visuals from coordinates, then summarizes a team's scoring possessions as an interactive average path and heatmap.

Default team: `glory`.

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd

from ufa import (
    average_scoring_path,
    build_scoring_possessions,
    cluster_scoring_possessions,
    fetch_shownspace_games,
    fetch_shownspace_season_throws,
    fetch_shownspace_throws_for_games,
    plot_average_scoring_path,
    plot_possession_path,
    plot_representative_paths,
    plot_scoring_heatmap,
    select_representative_paths,
    select_top_paths,
    summarize_path_clusters,
)

## Settings

Use `MAX_GAMES = 3` first to validate the visual quickly. Set `MAX_GAMES = None` for the full team season.

In [ ]:
SEASON = 2026
TEAM_ID = "glory"
MAX_GAMES = 3
SAMPLE_GAMES_RANDOMLY = True
RANDOM_STATE = 7
UNIQUE_REPRESENTATIVE_GAMES = True
PULL_RECEIVE_SCORES_ONLY = True
LONG_FIELD_ONLY = True
MAX_START_Y = 45
MIN_FIELD_PROGRESS = 50

all_games = fetch_shownspace_games(season=SEASON, final_only=True)
team_games = all_games[
    all_games["HomeTeamID"].str.lower().eq(TEAM_ID.lower())
    | all_games["AwayTeamID"].str.lower().eq(TEAM_ID.lower())
].reset_index(drop=True)

if MAX_GAMES is None:
    games = team_games.copy()
elif SAMPLE_GAMES_RANDOMLY:
    games = (
        team_games
        .sample(n=min(MAX_GAMES, len(team_games)), random_state=RANDOM_STATE)
        .sort_values("StartTimestamp")
        .reset_index(drop=True)
    )
else:
    games = team_games.head(MAX_GAMES).copy()

throws = fetch_shownspace_throws_for_games(games["GameID"].tolist(), delay=0.15)

games[["GameID", "AwayTeamID", "HomeTeamID", "AwayScore", "HomeScore", "Status", "StartTimestamp"]]


In [3]:
possessions, paths = build_scoring_possessions(throws, team_id=TEAM_ID)

print(f"Throws loaded: {len(throws):,}")
print(f"Scoring possessions found for {TEAM_ID}: {len(possessions):,}")

possessions.sort_values("risk_adjusted_aec_per_throw", ascending=False).head(10)

Throws loaded: 1,582
Scoring possessions found for glory: 66


,possession_id,GameID,team_id,game_quarter,quarter_point,possession_num,is_home_team,throw_count,total_aec,aec_per_throw,mean_cp,risk_adjusted_aec_per_throw,total_yards,yards_per_throw,max_throw_distance,huck_count,reset_count,lateral_yards
8,2026-04-25-DC-BOS|2|2|2|True,2026-04-25-DC-BOS,glory,2,2,2,True,3,1.669016,0.556339,0.975657,0.542796,15.810,5.270000,7.710785,0,0,13.61
0,2026-04-25-DC-BOS|1|1|2|True,2026-04-25-DC-BOS,glory,1,1,2,True,2,1.010065,0.505033,0.948091,0.478817,27.090,13.545000,21.883119,0,0,18.19
63,2026-05-16-BOS-MTL|5|1|2|False,2026-05-16-BOS-MTL,glory,5,1,2,False,2,1.000779,0.500390,0.951064,0.475903,16.260,8.130000,18.243903,0,1,19.73
32,2026-05-02-MTL-BOS|3|4|1|True,2026-05-02-MTL-BOS,glory,3,4,1,True,2,0.999607,0.499803,0.745103,0.372405,91.420,45.710000,84.106267,1,0,22.96
65,2026-05-16-BOS-MTL|5|4|2|False,2026-05-16-BOS-MTL,glory,5,4,2,False,3,1.025086,0.341695,0.936082,0.319855,9.100,3.033333,23.652150,0,0,45.92
24,2026-05-02-MTL-BOS|1|8|1|True,2026-05-02-MTL-BOS,glory,1,8,1,True,3,1.000000,0.333333,0.888191,0.296064,67.160,22.386667,47.421080,1,0,6.00
54,2026-05-16-BOS-MTL|2|10|1|False,2026-05-16-BOS-MTL,glory,2,10,1,False,4,1.013090,0.253273,0.886457,0.224515,89.870,22.467500,53.368539,1,0,24.37
29,2026-05-02-MTL-BOS|2|7|2|True,2026-05-02-MTL-BOS,glory,2,7,2,True,4,0.936796,0.234199,0.941492,0.220497,66.710,16.677500,29.638087,0,0,20.70
10,2026-04-25-DC-BOS|2|6|1|True,2026-04-25-DC-BOS,glory,2,6,1,True,4,1.000987,0.250247,0.873615,0.218619,93.670,23.417500,62.029739,1,0,35.72
58,2026-05-16-BOS-MTL|3|7|3|False,2026-05-16-BOS-MTL,glory,3,7,3,False,4,1.000000,0.250000,0.838444,0.209611,98.325,24.581250,55.371434,1,1,42.49


## Long-Field Possession Filter

Use this to focus the visuals on possessions that start farther from the scoring end zone, instead of short-field scores after turnovers.

In [ ]:
analysis_possessions = possessions.copy()

if PULL_RECEIVE_SCORES_ONLY:
    analysis_possessions = analysis_possessions[
        analysis_possessions["possession_num"].eq(1)
    ].copy()

if LONG_FIELD_ONLY:
    analysis_possessions = analysis_possessions[
        analysis_possessions["start_y"].le(MAX_START_Y)
        & analysis_possessions["field_progress"].ge(MIN_FIELD_PROGRESS)
    ].copy()

analysis_ids = set(analysis_possessions["possession_id"])
analysis_paths = [
    path for path in paths
    if path["possession_id"].iloc[0] in analysis_ids
]

print(f"All scoring possessions: {len(possessions):,}")
initial_scoring_holds = possessions["possession_num"].eq(1).sum()
print(f"Initial-possession scoring holds: {initial_scoring_holds:,}")
print(f"Analysis possessions: {len(analysis_possessions):,}")

analysis_possessions[[
    "possession_id", "GameID", "possession_num", "start_y", "end_y",
    "field_progress", "throw_count", "total_aec", "aec_per_throw"
]].head(10)


## Average Scoring Path

The average path is progress-normalized. Each scoring possession is resampled to fixed progress checkpoints from possession start to goal, then the checkpoint coordinates are averaged.

In [ ]:
avg_path = average_scoring_path(paths)
avg_path

In [ ]:
fig = plot_average_scoring_path(
    avg_path,
    paths=paths,
    title=f"{TEAM_ID.title()} average scoring path, {SEASON} sample",
    show_individual_paths=True,
)
fig.show()

## Real Representative Paths

The mean path is useful as a center-of-gravity check, but it can hide the actual bends, resets, hucks, and lateral movement that make an offense interesting. These cells keep real possessions intact and then pick examples worth studying.

In [ ]:
clustered_possessions = cluster_scoring_possessions(analysis_possessions, n_clusters=4)
cluster_summary = summarize_path_clusters(clustered_possessions)
cluster_summary

In [ ]:
# Try to show each representative path from a different game when possible.
representative_paths = select_representative_paths(
    clustered_possessions,
    analysis_paths,
    group_column="path_cluster",
    unique_games=UNIQUE_REPRESENTATIVE_GAMES,
)

rep_fig = plot_representative_paths(
    representative_paths,
    title=f"{TEAM_ID.title()} representative scoring path styles, {SEASON} sample",
)
rep_fig.show()

In [ ]:
top_paths = select_top_paths(
    clustered_possessions,
    analysis_paths,
    metric="aec_per_throw",
    n=3,
)

best_fig = plot_possession_path(
    top_paths[0],
    title=f"{TEAM_ID.title()} highest long-field aEC per throw scoring possession, {SEASON} sample",
)
best_fig.show()

## Catch Location Heatmap

This shows where completed throws in scoring possessions are caught.

In [6]:
heatmap = plot_scoring_heatmap(
    analysis_paths,
    title=f"{TEAM_ID.title()} scoring-possession catch heatmap, {SEASON} sample",
)
heatmap.show()

## Full Team Season

After the sample plots look right, run the full team season by setting `MAX_GAMES = None` below.

In [7]:
MAX_GAMES = None

games_full, throws_full = fetch_shownspace_season_throws(
    season=SEASON,
    team_id=TEAM_ID,
    max_games=MAX_GAMES,
    delay=0.15,
)

possessions_full, paths_full = build_scoring_possessions(throws_full, team_id=TEAM_ID)
avg_path_full = average_scoring_path(paths_full)

print(f"Games loaded: {len(games_full):,}")
print(f"Throws loaded: {len(throws_full):,}")
print(f"Scoring possessions found for {TEAM_ID}: {len(possessions_full):,}")

possessions_full.sort_values("risk_adjusted_aec_per_throw", ascending=False).head(20)

Games loaded: 10
Throws loaded: 5,528
Scoring possessions found for glory: 242


,possession_id,GameID,team_id,game_quarter,quarter_point,possession_num,is_home_team,throw_count,total_aec,aec_per_throw,mean_cp,risk_adjusted_aec_per_throw,total_yards,yards_per_throw,max_throw_distance,huck_count,reset_count,lateral_yards
90,2026-05-24-PHI-BOS|4|10|2|True,2026-05-24-PHI-BOS,glory,4,10,2,True,1,1.000000,1.000000,0.981701,0.981701,1.74,1.740,2.510717,0,0,1.81
92,2026-05-24-PHI-BOS|4|12|2|True,2026-05-24-PHI-BOS,glory,4,12,2,True,1,1.000000,1.000000,0.976720,0.976720,4.06,4.060,6.068253,0,0,4.51
164,2026-06-12-BOS-NY|2|4|2|False,2026-06-12-BOS-NY,glory,2,4,2,False,1,1.000000,1.000000,0.970723,0.970723,5.77,5.770,7.767426,0,0,5.20
129,2026-06-05-NY-BOS|3|8|2|True,2026-06-05-NY-BOS,glory,3,8,2,True,1,1.000000,1.000000,0.966872,0.966872,2.60,2.600,9.724037,0,0,9.37
91,2026-05-24-PHI-BOS|4|11|2|True,2026-05-24-PHI-BOS,glory,4,11,2,True,1,1.000000,1.000000,0.966484,0.966484,8.90,8.900,10.287643,0,0,5.16
106,2026-05-31-BOS-TOR|3|4|3|False,2026-05-31-BOS-TOR,glory,3,4,3,False,1,1.000000,1.000000,0.962996,0.962996,12.58,12.580,12.600020,0,0,0.71
135,2026-06-05-NY-BOS|4|8|3|True,2026-06-05-NY-BOS,glory,4,8,3,True,1,1.000000,1.000000,0.935900,0.935900,12.91,12.910,19.466715,0,0,14.57
93,2026-05-24-PHI-BOS|4|13|4|True,2026-05-24-PHI-BOS,glory,4,13,4,True,1,1.000000,1.000000,0.934068,0.934068,19.87,19.870,20.529065,0,0,5.16
197,2026-06-13-BOS-DC|3|6|2|False,2026-06-13-BOS-DC,glory,3,6,2,False,1,1.000000,1.000000,0.924361,0.924361,19.55,19.550,19.586184,0,0,1.19
8,2026-04-25-DC-BOS|2|2|2|True,2026-04-25-DC-BOS,glory,2,2,2,True,3,1.669016,0.556339,0.975657,0.542796,15.81,5.270,7.710785,0,0,13.61


In [8]:
fig_full = plot_average_scoring_path(
    avg_path_full,
    paths=paths_full,
    title=f"{TEAM_ID.title()} average scoring path, {SEASON}",
    show_individual_paths=True,
)
fig_full.show()

In [9]:
heatmap_full = plot_scoring_heatmap(
    paths_full,
    title=f"{TEAM_ID.title()} scoring-possession catch heatmap, {SEASON}",
)
heatmap_full.show()